# Vectorization & Performance

Topics:
- What is vectorization and why it's fast
- Vectorized operations vs Python loops
- `np.where()` — vectorized if/else
- `np.select()` — vectorized multiple conditions
- `.apply()` — when and when NOT to use it
- `.map()` — element-wise mapping
- `pd.cut()` and `pd.qcut()` — binning
- Profiling with `%%timeit`

In [3]:
import pandas as pd
import numpy as np

## 1. What is Vectorization?

Vectorization means applying an operation to an entire array at once instead of looping row by row.
Pandas and NumPy operations are vectorized — they run in optimized C code under the hood, not Python.

**Python loop** — one iteration at a time, slow:
```
for each row:          ← Python overhead per row
    do operation       ← slow
```

**Vectorized** — entire array at once, fast:
```
apply operation to all rows simultaneously  ← C speed
```

Rule: **never loop over rows when a vectorized alternative exists.**

In [4]:
np.random.seed(42)
df = pd.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve",
                   "Frank", "Grace", "Hank", "Iris", "Jack"],
    "salary":     np.random.randint(40000, 120000, size=10),
    "age":        np.random.randint(22, 60, size=10),
    "department": np.random.choice(["Eng", "HR", "Marketing"], size=10),
    "rating":     np.round(np.random.uniform(2.5, 5.0, size=10), 1),
    "years_exp":  np.random.randint(1, 20, size=10)
})

df

,name,salary,age,department,rating,years_exp
0,Alice,55795,43,Eng,4.5,18
1,Bob,40860,23,Eng,3.0,9
2,Carol,116820,45,Eng,3.8,2
3,Dave,94886,51,Marketing,4.0,15
4,Eve,46265,59,Marketing,2.6,7
5,Frank,77194,23,Marketing,4.0,12
6,Grace,84131,42,HR,2.9,8
7,Hank,100263,54,Marketing,2.7,15
8,Iris,56023,33,HR,4.9,3
9,Jack,81090,43,HR,4.9,14


## 2. Loop vs Vectorized — Speed Comparison

In [5]:
# create a larger DataFrame for benchmarking
big_df = pd.DataFrame({
    "salary": np.random.randint(40000, 120000, size=100_000)
})

In [6]:
%%timeit
# SLOW — Python loop
result = []
for salary in big_df["salary"]:
    result.append(salary * 1.1)

8.12 ms ± 184 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
%%timeit
# FAST — vectorized
result = big_df["salary"] * 1.1

99.5 μs ± 4.19 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 3. Basic Vectorized Operations

Any arithmetic on a Series/column is automatically vectorized.
You never need a loop for math.

In [8]:
# all of these are vectorized — applied to every row simultaneously
df["salary_after_tax"] = df["salary"] * 0.8
df["bonus"]            = df["salary"] * (df["rating"] / 5)
df["total_comp"]       = df["salary"] + df["bonus"]

df[["name", "salary", "rating", "bonus", "total_comp"]]

,name,salary,rating,bonus,total_comp
0,Alice,55795,4.5,50215.50,106010.50
1,Bob,40860,3.0,24516.00,65376.00
2,Carol,116820,3.8,88783.20,205603.20
3,Dave,94886,4.0,75908.80,170794.80
4,Eve,46265,2.6,24057.80,70322.80
5,Frank,77194,4.0,61755.20,138949.20
6,Grace,84131,2.9,48795.98,132926.98
7,Hank,100263,2.7,54142.02,154405.02
8,Iris,56023,4.9,54902.54,110925.54
9,Jack,81090,4.9,79468.20,160558.20


In [9]:
# string operations are vectorized too via .str accessor
df["name_upper"] = df["name"].str.upper()
df["name_len"]   = df["name"].str.len()

df[["name", "name_upper", "name_len"]]

,name,name_upper,name_len
0,Alice,ALICE,5
1,Bob,BOB,3
2,Carol,CAROL,5
3,Dave,DAVE,4
4,Eve,EVE,3
5,Frank,FRANK,5
6,Grace,GRACE,5
7,Hank,HANK,4
8,Iris,IRIS,4
9,Jack,JACK,4


## 4. `np.where()` — Vectorized If/Else

`np.where(condition, value_if_true, value_if_false)`

The vectorized equivalent of:
```python
if condition:
    value_if_true
else:
    value_if_false
```
Applied to every row simultaneously — no loop needed.

In [10]:
# label employees as high/low earners
df["salary_band"] = np.where(df["salary"] >= 80000, "High", "Low")

df[["name", "salary", "salary_band"]]


,name,salary,salary_band
0,Alice,55795,Low
1,Bob,40860,Low
2,Carol,116820,High
3,Dave,94886,High
4,Eve,46265,Low
5,Frank,77194,Low
6,Grace,84131,High
7,Hank,100263,High
8,Iris,56023,Low
9,Jack,81090,High


In [11]:
# can also use expressions as values
df["adjusted_salary"] = np.where(
    df["rating"] >= 4.0,
    df["salary"] * 1.10,   # 10% raise for high performers
    df["salary"] * 1.02    # 2% raise for others
)

df[["name", "salary", "rating", "adjusted_salary"]]

,name,salary,rating,adjusted_salary
0,Alice,55795,4.5,61374.50
1,Bob,40860,3.0,41677.20
2,Carol,116820,3.8,119156.40
3,Dave,94886,4.0,104374.60
4,Eve,46265,2.6,47190.30
5,Frank,77194,4.0,84913.40
6,Grace,84131,2.9,85813.62
7,Hank,100263,2.7,102268.26
8,Iris,56023,4.9,61625.30
9,Jack,81090,4.9,89199.00


## 5. `np.select()` — Vectorized Multiple Conditions

When you have more than 2 cases, `np.where()` becomes nested and unreadable.
`np.select()` handles multiple conditions cleanly.

```python
np.select(conditions, choices, default)
```
- `conditions` — list of boolean conditions (checked in order)
- `choices` — list of values (matched to conditions)
- `default` — value if no condition matches

In [12]:
# classify salary into 3 bands
conditions = [
    df["salary"] >= 100000,
    df["salary"] >= 70000,
    df["salary"] >= 40000
]

choices = ["Senior", "Mid", "Junior"]

df["level"] = np.select(conditions, choices, default="Unknown")

df[["name", "salary", "level"]]


,name,salary,level
0,Alice,55795,Junior
1,Bob,40860,Junior
2,Carol,116820,Senior
3,Dave,94886,Mid
4,Eve,46265,Junior
5,Frank,77194,Mid
6,Grace,84131,Mid
7,Hank,100263,Senior
8,Iris,56023,Junior
9,Jack,81090,Mid


In [13]:
# performance tier based on both rating and years_exp
conditions = [
    (df["rating"] >= 4.1) & (df["years_exp"] >= 10),
    (df["rating"] >= 3.5) & (df["years_exp"] >= 5),
    (df["rating"] >= 2.7) & (df["years_exp"] >= 3)
]

choices = ["Star", "Strong", "Average"]

df["performance"] = np.select(conditions, choices, default="Needs Improvement")

df[["name", "rating", "years_exp", "performance"]]

,name,rating,years_exp,performance
0,Alice,4.5,18,Star
1,Bob,3.0,9,Average
2,Carol,3.8,2,Needs Improvement
3,Dave,4.0,15,Strong
4,Eve,2.6,7,Needs Improvement
5,Frank,4.0,12,Strong
6,Grace,2.9,8,Average
7,Hank,2.7,15,Average
8,Iris,4.9,3,Average
9,Jack,4.9,14,Star


## 6. `.apply()` — When to Use and When NOT To

`.apply()` runs a function row by row (or column by column) — it's essentially a loop.
It's flexible but slow. Use it only when no vectorized alternative exists.

**Use `.apply()` when:**
- Logic is too complex for `np.where` / `np.select`
- Calling external functions or APIs per row
- Working with multiple columns together in complex ways

**Do NOT use `.apply()` when:**
- Simple math → use vectorized arithmetic
- If/else → use `np.where()` or `np.select()`
- String operations → use `.str` accessor

In [14]:
# BAD — using apply for something vectorized can do
df["salary_x2_slow"] = df["salary"].apply(lambda x: x * 2)

# GOOD — vectorized
df["salary_x2_fast"] = df["salary"] * 2

In [15]:
# apply on a single column — function receives one value at a time
def salary_grade(salary):
    if salary >= 100000:
        return "A"
    elif salary >= 70000:
        return "B"
    else:
        return "C"

df["grade"] = df["salary"].apply(salary_grade)
df[["name", "salary", "grade"]]

,name,salary,grade
0,Alice,55795,C
1,Bob,40860,C
2,Carol,116820,A
3,Dave,94886,B
4,Eve,46265,C
5,Frank,77194,B
6,Grace,84131,B
7,Hank,100263,A
8,Iris,56023,C
9,Jack,81090,B


In [16]:
# apply with axis=1 — function receives one ROW at a time (as a Series) 
# useful when logic involves multiple columns
def comp_score(row):
    return round((row["salary"] / 1000) * row["rating"] * (1 + row["years_exp"] / 100), 2)

df["comp_score"] = df.apply(comp_score, axis=1)
df[["name", "salary", "rating", "years_exp", "comp_score"]]

,name,salary,rating,years_exp,comp_score
0,Alice,55795,4.5,18,296.27
1,Bob,40860,3.0,9,133.61
2,Carol,116820,3.8,2,452.79
3,Dave,94886,4.0,15,436.48
4,Eve,46265,2.6,7,128.71
5,Frank,77194,4.0,12,345.83
6,Grace,84131,2.9,8,263.50
7,Hank,100263,2.7,15,311.32
8,Iris,56023,4.9,3,282.75
9,Jack,81090,4.9,14,452.97


```python

axis	            function receives
axis=0 (default)	one column at a time
axis=1	            one row at a time


## 7. `.map()` — Element-wise Mapping

`.map()` replaces values in a Series using a dictionary or function.
Cleaner than `.apply()` for simple substitutions.

In [17]:
# map department to a numeric code
dept_map = {"Eng": 1, "HR": 2, "Marketing": 3}

df["dept_code"] = df["department"].map(dept_map)
df[["name", "department", "dept_code"]]


,name,department,dept_code
0,Alice,Eng,1
1,Bob,Eng,1
2,Carol,Eng,1
3,Dave,Marketing,3
4,Eve,Marketing,3
5,Frank,Marketing,3
6,Grace,HR,2
7,Hank,Marketing,3
8,Iris,HR,2
9,Jack,HR,2


In [18]:
# map with a function — same as apply on a Series
df["salary_k"] = df["salary"].map(lambda x: f"${x//1000}k")
df[["name", "salary", "salary_k"]]

,name,salary,salary_k
0,Alice,55795,$55k
1,Bob,40860,$40k
2,Carol,116820,$116k
3,Dave,94886,$94k
4,Eve,46265,$46k
5,Frank,77194,$77k
6,Grace,84131,$84k
7,Hank,100263,$100k
8,Iris,56023,$56k
9,Jack,81090,$81k


## 8. `pd.cut()` and `pd.qcut()` — Binning

Binning divides a continuous column into discrete buckets.

- `pd.cut()` — bins based on **value ranges** (you define the edges)
- `pd.qcut()` — bins based on **quantiles** (equal number of rows per bin)

In [19]:
# pd.cut — fixed value ranges
df["salary_bin"] = pd.cut(
    df["salary"],
    bins   = [0, 60000, 80000, 100000, 200000],
    labels = ["Low", "Mid", "High", "Very High"]
)

df[["name", "salary", "salary_bin"]]

,name,salary,salary_bin
0,Alice,55795,Low
1,Bob,40860,Low
2,Carol,116820,Very High
3,Dave,94886,High
4,Eve,46265,Low
5,Frank,77194,Mid
6,Grace,84131,High
7,Hank,100263,Very High
8,Iris,56023,Low
9,Jack,81090,High


In [20]:
# pd.qcut — equal-sized groups (quartiles)
df["salary_quartile"] = pd.qcut(
    df["salary"],
    q      = 4,
    labels = ["Q1", "Q2", "Q3", "Q4"]
)

df[["name", "salary", "salary_quartile"]]

,name,salary,salary_quartile
0,Alice,55795,Q1
1,Bob,40860,Q1
2,Carol,116820,Q4
3,Dave,94886,Q4
4,Eve,46265,Q1
5,Frank,77194,Q2
6,Grace,84131,Q3
7,Hank,100263,Q4
8,Iris,56023,Q2
9,Jack,81090,Q3


In [21]:
# count employees per bin
print("cut:",  df["salary_bin"].value_counts())
print()
print("qcut:", df["salary_quartile"].value_counts())

cut: salary_bin
Low          4
High         3
Very High    2
Mid          1
Name: count, dtype: int64

qcut: salary_quartile
Q1    3
Q4    3
Q2    2
Q3    2
Name: count, dtype: int64


## 9. Profiling with `%%timeit`

`%%timeit` is a Jupyter magic command that runs the cell multiple times and reports average execution time.
Use it to compare approaches and find bottlenecks.

In [22]:
big = pd.DataFrame({"salary": np.random.randint(40000, 120000, size=500_000)})

In [23]:
%%timeit
# apply — slow
big["salary"].apply(lambda x: "High" if x >= 80000 else "Low")

67.3 ms ± 747 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [24]:
%%timeit
# np.where — fast
np.where(big["salary"] >= 80000, "High", "Low")

1.08 ms ± 47.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 10. Summary — When to Use What

| Task | Best approach |
|------|---------------|
| Math on a column | Vectorized arithmetic (`df["col"] * 2`) |
| Single if/else | `np.where()` |
| Multiple conditions | `np.select()` |
| Replace values with a dict | `.map()` |
| Complex logic across columns | `.apply(axis=1)` |
| Bin by value range | `pd.cut()` |
| Bin by equal groups | `pd.qcut()` |
| Never | Python `for` loop over rows |

## Practice Exercises

1. Add a column `tax` — 30% of salary if salary >= 80000, else 20%  (use `np.where`)
2. Add a column `experience_band` — "Junior" (< 3 yrs), "Mid" (3–8 yrs), "Senior" (> 8 yrs) (use `np.select`)
3. Add a column `dept_label` — map: Eng → Engineering, HR → Human Resources, Marketing → Marketing (use `.map`)
4. Bin `age` into 3 equal-sized groups using `pd.qcut()`, label them "Young", "Mid", "Senior"
5. Use `%%timeit` to compare `.apply(lambda x: x*1.1)` vs vectorized `* 1.1` on a 500k row DataFrame

In [25]:
# Question 1:
df["tax"] = np.where(df["salary"] >= 80000, df["salary"] * 0.3, df["salary"] * 0.2)


In [26]:
# question 2:
df["experience_band"] = np.select(
    [
        df["years_exp"] > 8,
        df["years_exp"] >= 3,
        (df["years_exp"] >= 0) & (df["years_exp"] < 3),
    ],
    ["Senior", "Mid", "Junior"],
    default="Unknown"
)


In [27]:
# question 3:
dep_label = {"Eng": "Engineering", "HR": "Human Resources", "Marketing": "Marketing"}

df["dept_label"] = df["department"].map(dep_label)

In [28]:
# question 4:
df["age_label"] = pd.qcut(
    df["age"],
    q = 3,
    labels = ["Young", "Mid", "Senior"]
)

In [29]:

%%timeit
# question 5:
# using apply
big["salary_with_bonus"] = big["salary"].apply(lambda x: x * 1.1)

70.7 ms ± 890 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [30]:
%%timeit

# using vectorized approach
big["salary_with_bonus"] = big["salary"] * 1.1

759 μs ± 21.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [31]:
print(df.head())

    name  salary  age department  rating  years_exp  salary_after_tax  \
0  Alice   55795   43        Eng     4.5         18           44636.0   
1    Bob   40860   23        Eng     3.0          9           32688.0   
2  Carol  116820   45        Eng     3.8          2           93456.0   
3   Dave   94886   51  Marketing     4.0         15           75908.8   
4    Eve   46265   59  Marketing     2.6          7           37012.0   

     bonus  total_comp name_upper  ...  grade comp_score  dept_code salary_k  \
0  50215.5    106010.5      ALICE  ...      C     296.27          1     $55k   
1  24516.0     65376.0        BOB  ...      C     133.61          1     $40k   
2  88783.2    205603.2      CAROL  ...      A     452.79          1    $116k   
3  75908.8    170794.8       DAVE  ...      B     436.48          3     $94k   
4  24057.8     70322.8        EVE  ...      C     128.71          3     $46k   

  salary_bin  salary_quartile      tax experience_band   dept_label  age_label  

In [32]:
print(big.head())

   salary  salary_with_bonus
0   76044            83648.4
1   89822            98804.2
2   58586            64444.6
3   62796            69075.6
4   59001            64901.1
